In [1]:
import os
import shutil
import random
from glob import glob
from ultralytics import YOLO
import yaml

# --- НАЛАШТУВАННЯ ---
SOURCE_LABELS_DIR = 'lpr_datasets/yolo/labels'
SOURCE_IMAGES_DIR = 'unity/images' # Де лежать всі твої оригінальні картинки

PROJECT_DIR = 'yolo_project' # Папка, де буде жити навчання
DATASET_DIR = os.path.join(PROJECT_DIR, 'dataset')

# Створюємо структуру папок, яку любить YOLO
for split in ['train', 'val']:
    os.makedirs(os.path.join(DATASET_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(DATASET_DIR, 'labels', split), exist_ok=True)

def prepare_dataset():
    print("--- Підготовка даних ---")
    label_files = glob(os.path.join(SOURCE_LABELS_DIR, '*.txt'))

    if not label_files:
        raise FileNotFoundError("Не знайдено txt файлів! Перевір шляхи.")

    # Перемішуємо
    random.shuffle(label_files)

    # Розбиваємо 90% на навчання, 10% на тест
    split_idx = int(len(label_files) * 0.9)
    train_files = label_files[:split_idx]
    val_files = label_files[split_idx:]

    print(f"Всього файлів: {len(label_files)}. Train: {len(train_files)}, Val: {len(val_files)}")

    def copy_files(files, split_name):
        for label_path in files:
            base_name = os.path.basename(label_path).replace('.txt', '')

            # Копіюємо лейбл
            shutil.copy(label_path, os.path.join(DATASET_DIR, 'labels', split_name))

            # Шукаємо і копіюємо картинку
            # Перевіряємо jpg та png
            img_src = os.path.join(SOURCE_IMAGES_DIR, base_name + '.jpg')
            ext = '.jpg'
            if not os.path.exists(img_src):
                img_src = os.path.join(SOURCE_IMAGES_DIR, base_name + '.png')
                ext = '.png'

            if os.path.exists(img_src):
                shutil.copy(img_src, os.path.join(DATASET_DIR, 'images', split_name, base_name + ext))
            else:
                print(f"Увага: картинку для {base_name} не знайдено!")

    copy_files(train_files, 'train')
    copy_files(val_files, 'val')

    # Створюємо data.yaml
    yaml_data = {
        'path': os.path.abspath(DATASET_DIR),
        'train': 'images/train',
        'val': 'images/val',
        'names': {0: 'number_zone'}
    }

    yaml_path = os.path.join(PROJECT_DIR, 'data.yaml')
    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_data, f)

    return yaml_path

def train_yolo(yaml_path):
    print("--- Починаємо навчання YOLO ---")
    # Завантажуємо найлегшу модель (nano)
    model = YOLO('yolov8n.pt')

    # Тренуємо
    # epochs=10 для перевірки, imgsz=640 - стандарт
    results = model.train(
        data=yaml_path,
        epochs=10,
        imgsz=640,
        project=os.path.join(PROJECT_DIR, 'runs'),
        name='exp_numbers',
        plots=True
    )
    print("Навчання завершено!")

if __name__ == '__main__':
    yaml_config = prepare_dataset()
    train_yolo(yaml_config)

--- Підготовка даних ---
Всього файлів: 6637. Train: 5973, Val: 664
--- Починаємо навчання YOLO ---
New https://pypi.org/project/ultralytics/8.3.233 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.224 🚀 Python-3.12.3 torch-2.9.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_project/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, 

/home/yesman/tfvenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 1.6±0.3 ms, read: 30.6±5.3 MB/s, size: 107.2 KB)
val: Scanning /mnt/c/Users/user/PycharmProjects/Diploma/try5/yolo_project/dataset/labels/val... 664 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 664/664 170.1it/s 3.9s0.1s
val: New cache created: /mnt/c/Users/user/PycharmProjects/Diploma/try5/yolo_project/dataset/labels/val.cache
Plotting labels to /mnt/c/Users/user/PycharmProjects/Diploma/try5/yolo_project/runs/exp_numbers/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 640 train, 640 v